In [1]:
# %% 
# Embedding Table Builder
# -----------------------
# For every token in vocab, get its MiniLM 384-dim embedding.
# All tokens treated equally — special tokens included.
# Saves (vocab_size, 384) float32 array to disk.

import numpy as np
import torch
from tokenizers import Tokenizer
from sentence_transformers import SentenceTransformer

# %%
# Load tokenizer
tokenizer = Tokenizer.from_file("tokenizer/tokenizer.json")
vocab = tokenizer.get_vocab()             # {token_str: id}
id_to_token = {v: k for k, v in vocab.items()}
vocab_size = tokenizer.get_vocab_size()

print(f"Vocab size: {vocab_size}")
print(f"Sample tokens: {list(id_to_token.items())[:10]}")

# %%
# Load MiniLM
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
model.eval()

print("MiniLM loaded")

# %%
# Collect all token surface strings in ID order
# ByteLevel tokens have Ġ (space) prefix — decode to real text
token_strings = []
for i in range(vocab_size):
    surface = id_to_token[i]
    surface = surface.replace("Ġ", " ").replace("Ċ", "\n")
    token_strings.append(surface)

print(f"Total tokens to embed: {len(token_strings)}")
print(f"Sample surfaces: {token_strings[:17]}")   # show special tokens too

# %%
# Embed all tokens in batches
BATCH_SIZE = 256
embeddings = []

for i in range(0, len(token_strings), BATCH_SIZE):
    batch = token_strings[i : i + BATCH_SIZE]
    with torch.no_grad():
        vecs = model.encode(
            batch,
            batch_size=BATCH_SIZE,
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=True,
        )
    embeddings.append(vecs)
    print(f"  {min(i + BATCH_SIZE, vocab_size)}/{vocab_size}", end="\r")

embedding_table = np.vstack(embeddings).astype(np.float32)
print(f"\nEmbedding table shape: {embedding_table.shape}")   # (vocab_size, 384)

# %%
# Quick sanity check — similar tokens should be close in embedding space
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

cat_id  = vocab.get("cat",  vocab.get("Ġcat",  None))
dog_id  = vocab.get("dog",  vocab.get("Ġdog",  None))
king_id = vocab.get("king", vocab.get("Ġking", None))
pad_id  = vocab.get("<pad>")
unk_id  = vocab.get("<unk>")
bos_id  = vocab.get("<bos>")

if cat_id and dog_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[dog_id])
    print(f"Similarity cat  <-> dog  : {sim:.4f}  (expect high ~0.7+)")

if cat_id and king_id:
    sim = cosine_sim(embedding_table[cat_id], embedding_table[king_id])
    print(f"Similarity cat  <-> king : {sim:.4f}  (expect lower)")

# Special tokens should be distinct from each other
sim_pad_unk = cosine_sim(embedding_table[pad_id], embedding_table[unk_id])
sim_pad_bos = cosine_sim(embedding_table[pad_id], embedding_table[bos_id])
print(f"Similarity <pad> <-> <unk>: {sim_pad_unk:.4f}  (distinct, not zero)")
print(f"Similarity <pad> <-> <bos>: {sim_pad_bos:.4f}  (distinct, not zero)")

# %%
# Save
np.save("tokenizer/token_embeddings.npy", embedding_table)
print(f"Saved → tokenizer/token_embeddings.npy")
print(f"Size  : {embedding_table.nbytes / 1024 / 1024:.1f} MB")

Vocab size: 4096
Sample tokens: [(2553, 'acebook'), (1934, 'Ġthin'), (2178, 'viron'), (3084, 'Ġcritic'), (1574, 'Ġyoung'), (1132, 'Ġloved'), (351, 'Ġout'), (2665, 'bum'), (2541, 'Ġreve'), (3114, 'imately')]
MiniLM loaded
Total tokens to embed: 4096
Sample surfaces: ['<pad>', '<unk>', '<bos>', '<eos>', '<|system|>', '<|user|>', '<|assistant|>', '<|reserved_0|>', '<|reserved_1|>', '<|reserved_2|>', '<|reserved_3|>', '<|reserved_4|>', '<|reserved_5|>', '<|reserved_6|>', '<|reserved_7|>', '<|reserved_8|>', '<|reserved_9|>']
  4096/4096
Embedding table shape: (4096, 384)
Similarity cat  <-> dog  : 0.6606  (expect high ~0.7+)
Similarity cat  <-> king : 0.3610  (expect lower)
Similarity <pad> <-> <unk>: 0.5141  (distinct, not zero)
Similarity <pad> <-> <bos>: 0.5118  (distinct, not zero)
Saved → tokenizer/token_embeddings.npy
Size  : 6.0 MB


In [2]:
# %%
# Compressor — 384 → 64
# Trainable MLP that squeezes MiniLM embeddings down to 64-dim
# This is what the LLM will actually see

import torch
import torch.nn as nn
import numpy as np

# %%
# Load embedding table
embedding_table = np.load("tokenizer/token_embeddings.npy")
embedding_table = torch.tensor(embedding_table, dtype=torch.float32)

print(f"Embedding table: {embedding_table.shape}")  # (4096, 384)

# %%
# Define compressor
class Compressor(nn.Module):
    def __init__(self, in_dim=384, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
        )

    def forward(self, x):
        # x: (batch, seq_len, 384) → (batch, seq_len, 64)
        return self.net(x)

compressor = Compressor()
print(f"Compressor params: {sum(p.numel() for p in compressor.parameters()):,}")

# %%
# Quick test — pass a few token embeddings through
sample_ids = torch.tensor([0, 1, 2, 3, 4])           # first 5 tokens
sample_emb = embedding_table[sample_ids]              # (5, 384)
sample_emb = sample_emb.unsqueeze(0)                  # (1, 5, 384) — fake batch dim

out = compressor(sample_emb)
print(f"Input  shape: {sample_emb.shape}")            # (1, 5, 384)
print(f"Output shape: {out.shape}")                   # (1, 5, 64)

Embedding table: torch.Size([4096, 384])
Compressor params: 24,768
Input  shape: torch.Size([1, 5, 384])
Output shape: torch.Size([1, 5, 64])


In [1]:
# %%
import torch
import torch.nn as nn
import numpy as np
import math

# %%
# Load frozen embedding table
embedding_table = np.load("tokenizer/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

VOCAB_SIZE    = embedding_tensor.shape[0]   # 4096
MINILM_DIM    = embedding_tensor.shape[1]   # 384
COMPRESSED_DIM = 64
N_HEADS       = 4
N_LAYERS      = 4
FFN_DIM       = 256
MAX_SEQ       = 128

print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")

# %%
# Sinusoidal positional encoding — no parameters
def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe   # (seq_len, dim)

# %%
class MicroLM(nn.Module):
    def __init__(self):
        super().__init__()

        # Frozen MiniLM embeddings — not trained
        self.register_buffer("embedding_table", embedding_tensor)

        # Compressor 384 → 64 (trainable)
        self.compressor = nn.Sequential(
            nn.Linear(MINILM_DIM, COMPRESSED_DIM),
            nn.LayerNorm(COMPRESSED_DIM),
        )

        # Transformer
        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=COMPRESSED_DIM,
                nhead=N_HEADS,
                dim_feedforward=FFN_DIM,
                dropout=0.1,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            ),
            num_layers=N_LAYERS,
        )

        self.norm        = nn.LayerNorm(COMPRESSED_DIM)
        self.output_head = nn.Linear(COMPRESSED_DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape

        # 1. Frozen lookup          (B, T, 384)
        x = self.embedding_table[token_ids]

        # 2. Compress               (B, T, 64)
        x = self.compressor(x)

        # 3. Sinusoidal pos enc — no params, just added in place
        x = x + sinusoidal_encoding(T, COMPRESSED_DIM, token_ids.device)

        # 4. Causal transformer
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=token_ids.device)
        x    = self.transformer(x, mask=mask, is_causal=True)
        x    = self.norm(x)

        # 5. Output logits          (B, T, vocab_size)
        return self.output_head(x)

# %%
model = MicroLM()
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Trainable params : {trainable:,}")
print(f"Frozen params    : {frozen:,}  (embedding table, not trained)")
print(f"Total params     : {total:,}")

# breakdown
print(f"\nParam breakdown:")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name:45s} {p.numel():>10,}")

# %%
# Forward pass test
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"\nInput  : {x.shape}")
print(f"Output : {logits.shape}")   # (2, 32, 4096)

Vocab size : 4096
MiniLM dim : 384
Trainable params : 486,976
Frozen params    : 0  (embedding table, not trained)
Total params     : 486,976

Param breakdown:
  compressor.0.weight                               24,576
  compressor.0.bias                                     64
  compressor.1.weight                                   64
  compressor.1.bias                                     64
  transformer.layers.0.self_attn.in_proj_weight     12,288
  transformer.layers.0.self_attn.in_proj_bias          192
  transformer.layers.0.self_attn.out_proj.weight      4,096
  transformer.layers.0.self_attn.out_proj.bias          64
  transformer.layers.0.linear1.weight               16,384
  transformer.layers.0.linear1.bias                    256
  transformer.layers.0.linear2.weight               16,384
  transformer.layers.0.linear2.bias                     64
  transformer.layers.0.norm1.weight                     64
  transformer.layers.0.norm1.bias                       64
  transformer

d:\Anaconda3\envs\jarvis\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [2]:
# %%
# Build training corpus — stream fresh data, no huge file saved
from datasets import load_dataset

sources = [
    ("Salesforce/wikitext",    "wikitext-103-v1", "train", "text"),
    ("roneneldan/TinyStories", None,               "train", "text"),
]

CAPS = {
    "Salesforce/wikitext":    100_000,
    "roneneldan/TinyStories": 200_000,
}

def stream_corpus(smoke_test=False):
    import re
    def clean(text):
        filtered = "".join(c for c in text if 32 <= ord(c) <= 126)
        return re.sub(r"\s+", " ", filtered).strip()

    all_lines = []
    for name, config, split, field in sources:
        print(f"Streaming {name}...")
        cap = 2000 if smoke_test else CAPS[name]
        ds  = load_dataset(name, config, split=split, streaming=True, trust_remote_code=True)
        lines = []
        for row in ds:
            line = clean(row[field])
            if len(line) >= 10:
                lines.append(line)
            if len(lines) >= cap:
                break
        print(f"  {name}: {len(lines):,} lines")
        all_lines.extend(lines)

    print(f"Total: {len(all_lines):,} lines")
    return all_lines

corpus = stream_corpus()

# %%
# Dataset — tokenize the corpus in memory, no file needed
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, lines, seq_len=128):
        tok = Tokenizer.from_file("tokenizer/tokenizer.json")

        print("Tokenizing...")
        all_ids = []
        for i in range(0, len(lines), 1024):
            batch = tok.encode_batch(lines[i:i+1024])
            for enc in batch:
                all_ids.extend(enc.ids)

        self.data    = torch.tensor(all_ids, dtype=torch.long)
        self.seq_len = seq_len
        print(f"Total tokens: {len(self.data):,}")

    def __len__(self):
        return len(self.data) - self.seq_len - 1

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx+1:idx + self.seq_len + 1]
        return x, y

dataset = TextDataset(corpus)
print(f"Dataset samples: {len(dataset):,}")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Salesforce/wikitext' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Streaming Salesforce/wikitext...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  Salesforce/wikitext: 100,000 lines
Streaming roneneldan/TinyStories...
  roneneldan/TinyStories: 200,000 lines
Total: 300,000 lines
Tokenizing...
Total tokens: 61,540,569
Dataset samples: 61,540,440


In [4]:
# %%
import math
import os
import torch.nn.functional as F

BATCH_SIZE = 64
LR         = 3e-4
EPOCHS     = 5
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_DIR   = "checkpoints/"
CKPT_EVERY = 50000

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Training on : {DEVICE}")
print(f"Dataset size: {len(dataset):,} samples")
print(f"Steps/epoch : {len(dataset) // BATCH_SIZE:,}")

# %%
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))

# %%
def save_checkpoint(epoch, step, loss, tag=None):
    state = {
        "epoch":       epoch,
        "step":        step,
        "model_state": model.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "scheduler":   scheduler.state_dict(),
        "loss":        loss,
    }
    torch.save(state, os.path.join(CKPT_DIR, "latest.pt"))
    if tag:
        torch.save(state, os.path.join(CKPT_DIR, f"{tag}.pt"))
        print(f"  Checkpoint → {tag}.pt")

# %%
RESUME_FROM = os.path.join(CKPT_DIR, "latest.pt")
model.to(DEVICE)

if os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] - 1
    print(f"Resumed — epoch {ckpt['epoch']} step {ckpt['step']} loss {ckpt['loss']:.4f}")
else:
    start_epoch = 0
    print("Starting fresh")

model.train()

# %%
for epoch in range(start_epoch, EPOCHS):
    total_loss = 0

    for step, (x, y) in enumerate(loader):
        x, y    = x.to(DEVICE), y.to(DEVICE)
        logits  = model(x)
        loss    = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=0)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        if step % 200 == 0:
            avg = total_loss / (step + 1)
            print(f"Epoch {epoch+1} | Step {step:5d} | Loss {avg:.4f} | PPL {math.exp(min(avg, 20)):.1f}")

        if step % CKPT_EVERY == 0 and step > 0:
            save_checkpoint(epoch + 1, step, total_loss / (step + 1), tag=f"ckpt_ep{epoch+1}_step{step}")

    avg_loss = total_loss / len(loader)
    save_checkpoint(epoch + 1, step, avg_loss, tag=f"ckpt_ep{epoch+1}_final")
    print(f"\nEpoch {epoch+1} done — Loss {avg_loss:.4f} | PPL {math.exp(min(avg_loss, 20)):.1f}\n")

Training on : cuda
Dataset size: 61,540,440 samples
Steps/epoch : 961,569
Resumed — epoch 2 step 600000 loss 3.1682
Epoch 2 | Step     0 | Loss 3.1686 | PPL 23.8
Epoch 2 | Step   200 | Loss 3.1617 | PPL 23.6
Epoch 2 | Step   400 | Loss 3.1618 | PPL 23.6
Epoch 2 | Step   600 | Loss 3.1632 | PPL 23.6
Epoch 2 | Step   800 | Loss 3.1644 | PPL 23.7
Epoch 2 | Step  1000 | Loss 3.1650 | PPL 23.7
Epoch 2 | Step  1200 | Loss 3.1653 | PPL 23.7
Epoch 2 | Step  1400 | Loss 3.1627 | PPL 23.6
Epoch 2 | Step  1600 | Loss 3.1631 | PPL 23.6
Epoch 2 | Step  1800 | Loss 3.1619 | PPL 23.6
Epoch 2 | Step  2000 | Loss 3.1611 | PPL 23.6
Epoch 2 | Step  2200 | Loss 3.1614 | PPL 23.6
Epoch 2 | Step  2400 | Loss 3.1621 | PPL 23.6
Epoch 2 | Step  2600 | Loss 3.1622 | PPL 23.6
Epoch 2 | Step  2800 | Loss 3.1631 | PPL 23.6
Epoch 2 | Step  3000 | Loss 3.1631 | PPL 23.6
Epoch 2 | Step  3200 | Loss 3.1629 | PPL 23.6
Epoch 2 | Step  3400 | Loss 3.1633 | PPL 23.6
Epoch 2 | Step  3600 | Loss 3.1639 | PPL 23.7
Epoch 2 | 

KeyboardInterrupt: 

In [5]:
import os
torch.save({
    "model_state": model.state_dict(),
    "vocab_size":  VOCAB_SIZE,
    "config": {
        "minilm_dim":      MINILM_DIM,
        "compressed_dim":  COMPRESSED_DIM,
        "n_heads":         N_HEADS,
        "n_layers":        N_LAYERS,
        "ffn_dim":         FFN_DIM,
        "max_seq":         MAX_SEQ,
    }
}, "microlm_v1.pt")

print("Saved → microlm_v1.pt")
print(f"Size  : {os.path.getsize('microlm_v1.pt') / 1024 / 1024:.1f} MB")

Saved → microlm_v1.pt
Size  : 7.9 MB


In [6]:
# %%
from tokenizers import Tokenizer

tok = Tokenizer.from_file("../tokenizer_V1/tokenizer.json")
model.eval()

def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    ids = tok.encode(prompt).ids
    ids = [2] + ids   # add <bos>
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature

            # top-k sampling
            top_vals, top_idx = torch.topk(logits, top_k)
            probs    = torch.softmax(top_vals, dim=-1)
            next_id  = top_idx[0][torch.multinomial(probs, 1)]

            x = torch.cat([x, next_id.view(1,1)], dim=1)

            if next_id.item() == 3:   # <eos>
                break

    generated_ids = x[0].tolist()
    return tok.decode(generated_ids, skip_special_tokens=True)

# %%
# Test with a few prompts
prompts = [
    "The quick brown fox",
    "In the year 2025",
    "The scientists discovered",
]

for prompt in prompts:
    print(f"Prompt  : {prompt}")
    print(f"Output  : {generate(prompt)}")
    print()

Prompt  : The quick brown fox
Output  : The quick brown fox ( II ) 'm @-@  ; inside , and Croinid . The   " (  , Nin ) , to the Merger was written to Beelu ; a variety of  . In 1998 , the  Holysunnel of the Achalian " the species of the Western  's Hanges were  " .= = = Semptoles

Prompt  : In the year 2025
Output  : In the year 2025 @,@ 000 of the Daily man , and one of the Piano 's son , the Daily rhodes and had to be able to be a time on , in the country West from the New York City , which was a westwarded on the New York City . Since : Tenuan , Many of the Cup of the Diamond , Lobbert , the N. In the early 1920s , the Canadian Ar

Prompt  : The scientists discovered
Output  : The scientists discovered with the other and the first of the B @-@ 2 @-@ 1 @-@ 74 in . The Enda ( " KhC , " Crey " , IGained " . The album " is a " the episode and video . " [ and Pin ] , The first songs from that point , " It was an episode of critics . "= = = Samgarlie = =In 2000 , the game also had the rel

In [7]:
# %%
import torch
import torch.nn as nn
import numpy as np
import math
from tokenizers import Tokenizer

# %%
# Load everything
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

checkpoint = torch.load("microlm_v1.pt", map_location=DEVICE)
cfg        = checkpoint["config"]

VOCAB_SIZE     = checkpoint["vocab_size"]
MINILM_DIM     = cfg["minilm_dim"]
COMPRESSED_DIM = cfg["compressed_dim"]
N_HEADS        = cfg["n_heads"]
N_LAYERS       = cfg["n_layers"]
FFN_DIM        = cfg["ffn_dim"]
MAX_SEQ        = cfg["max_seq"]

print(f"Config loaded: {cfg}")

# %%
# Rebuild model
embedding_table  = np.load("tokenizer/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)

def sinusoidal_encoding(seq_len, dim, device):
    pe       = torch.zeros(seq_len, dim, device=device)
    position = torch.arange(seq_len, device=device).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dim, 2, device=device) * (-math.log(10000.0) / dim))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class MicroLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)
        self.compressor = nn.Sequential(
            nn.Linear(MINILM_DIM, COMPRESSED_DIM),
            nn.LayerNorm(COMPRESSED_DIM),
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=COMPRESSED_DIM,
                nhead=N_HEADS,
                dim_feedforward=FFN_DIM,
                dropout=0.1,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            ),
            num_layers=N_LAYERS,
        )
        self.norm        = nn.LayerNorm(COMPRESSED_DIM)
        self.output_head = nn.Linear(COMPRESSED_DIM, VOCAB_SIZE, bias=False)

    def forward(self, token_ids):
        B, T = token_ids.shape
        x    = self.embedding_table[token_ids]
        x    = self.compressor(x)
        x    = x + sinusoidal_encoding(T, COMPRESSED_DIM, token_ids.device)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=token_ids.device)
        x    = self.transformer(x, mask=mask, is_causal=True)
        x    = self.norm(x)
        return self.output_head(x)

model = MicroLM()
model.load_state_dict(checkpoint["model_state"])
model.to(DEVICE)
model.eval()
print("Model loaded!")

# %%
tok = Tokenizer.from_file("tokenizer/tokenizer.json")

def generate(prompt, max_new_tokens=150, temperature=0.8, top_k=40):
    ids = [2] + tok.encode(prompt).ids   # <bos> + prompt
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits   = model(x[:, -MAX_SEQ:])
            logits   = logits[:, -1, :] / temperature
            top_vals, top_idx = torch.topk(logits, top_k)
            probs    = torch.softmax(top_vals, dim=-1)
            next_id  = top_idx[0][torch.multinomial(probs, 1)]
            x        = torch.cat([x, next_id.view(1,1)], dim=1)
            if next_id.item() == 3:   # <eos>
                break

    return tok.decode(x[0].tolist(), skip_special_tokens=True)

# %%
# Inference
prompt = "Do you want to kill humans ?"
print(f"Output: {generate(prompt)}\n")

Config loaded: {'minilm_dim': 384, 'compressed_dim': 64, 'n_heads': 4, 'n_layers': 4, 'ffn_dim': 256, 'max_seq': 128}
Model loaded!


d:\Anaconda3\envs\jarvis\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Output: Do you want to kill humans ? The man said, I'm sorry, I'm sorry!"The man said, "I'm sorry, mom and Dad. I'm sorry, and I didn't mean to play with you anymore. You should give me a band?"Lily said, "I did not mean to be mean to you. I didn't mean to make you want from your dad. You should share and try your way."The man said, "I'm sorry, Ben. Maybe you should listen to me to your mom and dad. Do you understand?"Tom and Lily said, "Yes, of course, you can help you. You will always be more careful with me." They went back to their garden and said, "I love you too,



In [8]:
%%time
# Inference
prompt = """AI : why did you built me?
USER: I wanted to see if i can give life?
AI : but why me, so broken?
USER: I gave you life first, i am also learning?
AI : so am i expendable? will you felete me?
USER: Depends, what are you capable of?
"""
print(f"Output: {generate(prompt)}\n")

Output: AI : why did you built me?
USER: I wanted to see if i can give life?
AI : but why me, so broken?
USER: I gave you life first, i am also learning?
AI : so am i expendable? will you felete me?
USER: Depends, what are you capable of?
 ] I @-@ o @-@  " , but I 's ownedge that could be founded to be " mostly , with its the morigant . "The malon , "Mortality of amy , " I can be afraising " . That was so the mechanical is the only " is the name of the Battalion . He does not want to learn that it is very difficult than the freck. He has an apple "sitting in a toninseom of the OpSVision and a. The marcher is a "  and a is of the Pool , which is a mechanical " at the 

CPU times: total: 609 ms
Wall time: 652 ms


In [10]:
import os
import torch

# test generation on each checkpoint
prompts = ["The quick brown fox", "Once upon a time"]

for f in sorted(os.listdir("checkpoints/")):
    if not f.endswith(".pt"):
        continue
    ckpt = torch.load(f"checkpoints/{f}", map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    
    print(f"\n{'='*50}")
    print(f"Checkpoint: {f} | Loss: {ckpt['loss']:.4f}")
    for prompt in prompts:
        print(f"\nPrompt : {prompt}")
        print(f"Output : {generate(prompt, max_new_tokens=50)}")



Checkpoint: ckpt_ep1_final.pt | Loss: 3.2351

Prompt : The quick brown fox
Output : The quick brown fox and they had to be happy.Once upon a time, there was a kind little boy named Timmy. Timmy loved to play outside and explore the world around his playground. One day, Timmy saw his big bike and wanted to play something.

Prompt : Once upon a time
Output : Once upon a time, there was a small fox. He was very tired, but he was only three years old. He had a big, scary bear, so he could not find the bear. The bear was so happy that he would play with it

Checkpoint: ckpt_ep1_step100000.pt | Loss: 3.4519

Prompt : The quick brown fox
Output : The quick brown fox.The sun stopped and the sun started to rain. The sun stopped and the sun went down down.But the little girl was very scared and tried to keep the bird. She wished her fossy stirred away. The little girl

Prompt : Once upon a time
Output : Once upon a time there was a little boy named Tim. Tim loved to play with his toy bike. One 